# MLP model code

In [ ]:
"""
MLP hyperparameter sweep for multiclass classification.

What it does
------------
1) Sweeps over a manual grid of hyperparameters.
2) For EACH hyperparameter combo:
   - trains with early stopping on validation macro-F1
   - evaluates metrics on validation + test
   - writes ONE ROW immediately to a CSV (append mode) => crash-safe
3) If re-run after a crash:
   - loads existing CSV and SKIPS combos already completed (resume capability)
4) After sweep completes:
   - selects best params by SELECT_BEST_BY
   - retrains best model and saves ONLY that model + metadata
5) Also saves the latent representation of the best model on the test set
   using the post-BN + post-ReLU representation from the fc3 block.

Outputs
-------
- results_dir/
    - sweep_metrics.csv
    - sweep_log_<timestamp>.txt
    - best_model_<timestamp>/
        - mlp_state_dict.pt
        - metadata.json
        - test_outputs.npz
        - test_latent_post_bn_relu_fc3.pt
"""

import os
import sys
import json
import time
import copy
import hashlib
import random
import itertools
import contextlib
from datetime import datetime
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight


# =====================================================
# USER CONFIG
# =====================================================
DATA_PATH = "../Data/multiclass_data_no_SMOTE"  # Path where training_data.pt, validation_data.pt, test_data.pt are located
RESULTS_DIR = "../Results/results_mlp_hparam_sweep"  # Directory to save sweep results and best model

USE_BALANCED_WEIGHTS = False  # Whether to use class-balanced weights in the loss function (set to False since we're using SMOTE on train)
GLOBAL_SEED = 42

# Options: "val_macro_f1", "val_bal_acc", "val_macro_auc_ovr", "val_mcc"
SELECT_BEST_BY = "val_macro_f1"

HIDDEN_SIZES = (512, 256, 128)
DATALOADER_WORKERS = 0

CLASS_ID_TO_NAME = {
    0: "phosphate",
    1: "sulfate",
    2: "chloride",
    3: "nitrate",
    4: "carbonate",
}

# =====================================================
# PARAM GRID
# =====================================================
PARAM_GRID = {
    "lr": [0.001],
    "weight_decay": [1e-4],
    "dropout": [0.2],
    "batch_size": [128],
    "epochs": [200],
    "patience": [10],
    "use_bn": [True],
}

# =====================================================
# REPRODUCIBILITY
# =====================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(GLOBAL_SEED)


# =====================================================
# IO SETUP
# =====================================================
os.makedirs(RESULTS_DIR, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_PATH = os.path.join(RESULTS_DIR, f"sweep_log_{ts}.txt")
CSV_PATH = os.path.join(RESULTS_DIR, "sweep_metrics.csv")

BEST_MODEL_DIR = os.path.join(RESULTS_DIR, f"best_model_{ts}")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)


# =====================================================
# TEE LOGGER
# =====================================================
class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            try:
                s.flush()
            except Exception:
                pass


# =====================================================
# LOAD DATA
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

print("📦 Loading datasets...")
train = torch.load(os.path.join(DATA_PATH, "training_data.pt"), map_location="cpu")
val   = torch.load(os.path.join(DATA_PATH, "validation_data.pt"), map_location="cpu")
test  = torch.load(os.path.join(DATA_PATH, "test_data.pt"), map_location="cpu")
print("✅ Data loaded successfully!")

def to_np(t):
    return t.detach().cpu().numpy() if torch.is_tensor(t) else np.array(t)

X_train, y_train = to_np(train["X_train"]), to_np(train["y_train"]).ravel().astype(int)
X_val,   y_val   = to_np(val["X_val"]),     to_np(val["y_val"]).ravel().astype(int)
X_test,  y_test  = to_np(test["X_test"]),   to_np(test["y_test"]).ravel().astype(int)

classes = np.unique(y_train)
n_classes = len(classes)
input_size = X_train.shape[1]

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"Classes: {n_classes} | Train distribution: {np.bincount(y_train)}")
print(f"CSV (incremental): {CSV_PATH}")


# =====================================================
# CLASS WEIGHTS
# =====================================================
if USE_BALANCED_WEIGHTS:
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(n_classes),
        y=y_train
    )
    print("Class weights:", class_weights)
else:
    class_weights = None
    print("Using uniform weights (no class weighting)")


# =====================================================
# DATALOADER
# =====================================================
def make_loader(X, y, batch_size=256, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    ds = TensorDataset(X_t, y_t)
    pin = (device.type == "cuda")
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=DATALOADER_WORKERS,
        pin_memory=pin
    )


# =====================================================
# MODEL
# =====================================================
class MLP(nn.Module):
    def __init__(self, in_dim, h1, h2, h3, out_dim, p_drop=0.3, use_bn=True):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1) if use_bn else nn.Identity()

        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2) if use_bn else nn.Identity()

        self.fc3 = nn.Linear(h2, h3)
        self.bn3 = nn.BatchNorm1d(h3) if use_bn else nn.Identity()

        self.out = nn.Linear(h3, out_dim)
        self.drop = nn.Dropout(p_drop)

    def forward_features(self, x, apply_dropout=False):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = F.relu(x)

        if apply_dropout:
            x = self.drop(x)

        return x

    def forward(self, x):
        x = self.forward_features(x, apply_dropout=True)
        return self.out(x)


# =====================================================
# METRICS
# =====================================================
def safe_multiclass_auc(y_true, y_proba, average="macro"):
    try:
        return roc_auc_score(y_true, y_proba, multi_class="ovr", average=average)
    except Exception:
        return np.nan

def predict_proba(model, X, batch_size=1024):
    loader = make_loader(X, np.zeros(len(X), dtype=int), batch_size=batch_size, shuffle=False)
    model.eval()
    probs_all = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(probs)
    return np.vstack(probs_all)

def per_class_f1_dict(y_true, y_pred, n_classes):
    scores = f1_score(y_true, y_pred, average=None, labels=np.arange(n_classes))
    return {f"f1_c{i}": float(scores[i]) for i in range(n_classes)}

def compute_metrics_with_per_class(y_true, y_pred, y_proba, n_classes):
    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_pred))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))
    out["weighted_f1"] = float(f1_score(y_true, y_pred, average="weighted"))
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro"))
    out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    out["macro_auc_ovr"] = float(safe_multiclass_auc(y_true, y_proba, average="macro"))
    out["weighted_auc_ovr"] = float(safe_multiclass_auc(y_true, y_proba, average="weighted"))
    out.update(per_class_f1_dict(y_true, y_pred, n_classes))
    return out

def save_prediction_outputs(save_dir, y_true, y_pred, y_proba, class_names=None, prefix="test"):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{prefix}_outputs.npz")

    if class_names is None:
        class_names = np.array([f"C{i}" for i in range(y_proba.shape[1])], dtype=object)
    else:
        class_names = np.array(class_names, dtype=object)

    np.savez_compressed(
        save_path,
        y_true=np.asarray(y_true),
        y_pred=np.asarray(y_pred),
        y_proba=np.asarray(y_proba),
        class_names=class_names
    )
    print(f"Saved prediction outputs to: {save_path}")

def collect_mlp_predictions(model, loader, device):
    model.eval()

    all_true = []
    all_pred = []
    all_proba = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            proba = torch.softmax(logits, dim=1)
            pred = torch.argmax(proba, dim=1)

            all_true.append(yb.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            all_proba.append(proba.cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    y_proba = np.concatenate(all_proba)

    return y_true, y_pred, y_proba


# =====================================================
# LATENT SPACE EXTRACTION + SAVE
# =====================================================
def collect_latent_features(model, loader, device):
    """
    Collect post-BN + post-ReLU latent features from model.forward_features().
    """
    model.eval()

    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            emb = model.forward_features(xb, apply_dropout=False)
            all_embeddings.append(emb.cpu())
            all_labels.append(yb.cpu())

    embeddings = torch.cat(all_embeddings, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy().astype(int)

    return embeddings, labels

def save_latent_space_pt(
    model,
    X,
    y,
    device,
    save_path,
    batch_size=1024,
    is_synthetic=None,
    latent_name="post_bn_relu_fc3"
):
    """
    Save post-ReLU latent space outputs as a .pt file.
    """
    loader = make_loader(X, y, batch_size=batch_size, shuffle=False)
    embeddings, labels = collect_latent_features(
        model=model,
        loader=loader,
        device=device
    )

    save_obj = {
        "embeddings": torch.tensor(embeddings, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.long),
        "latent_name": latent_name,
    }

    if is_synthetic is not None:
        is_synthetic = np.asarray(is_synthetic).ravel().astype(int)
        if len(is_synthetic) != len(labels):
            raise ValueError("Length of is_synthetic must match number of samples.")
        save_obj["is_synthetic"] = torch.tensor(is_synthetic, dtype=torch.long)

    torch.save(save_obj, save_path)
    print(f"Saved latent space .pt file to: {save_path}")
    print(f"  latent_name : {latent_name}")
    print(f"  embeddings  : {save_obj['embeddings'].shape}")
    print(f"  labels      : {save_obj['labels'].shape}")
    if "is_synthetic" in save_obj:
        print(f"  is_synthetic: {save_obj['is_synthetic'].shape}")


# =====================================================
# PARAM UTILITIES
# =====================================================
def stable_param_id(params: dict) -> str:
    s = json.dumps(params, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]

def load_completed_param_ids(csv_path: str) -> set:
    if not os.path.exists(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path)
        if "param_id" not in df.columns:
            return set()
        return set(df["param_id"].astype(str).tolist())
    except Exception:
        return set()


# =====================================================
# TRAINING
# =====================================================
def train_one_model(X_tr, y_tr, X_va, y_va, params, class_weights=None, seed=42):
    seed_everything(seed)

    model = MLP(
        input_size,
        HIDDEN_SIZES[0], HIDDEN_SIZES[1], HIDDEN_SIZES[2],
        n_classes,
        p_drop=params["dropout"],
        use_bn=params["use_bn"]
    ).to(device)

    train_loader = make_loader(X_tr, y_tr, batch_size=params["batch_size"], shuffle=True)
    val_loader   = make_loader(X_va, y_va, batch_size=1024, shuffle=False)

    if class_weights is not None:
        cw = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=cw)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])

    best_val_macro_f1 = -1.0
    best_state = None
    epochs_no_improve = 0
    patience = int(params["patience"])

    for _epoch in range(1, int(params["epochs"]) + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        scheduler.step()

        model.eval()
        yv_pred = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                yv_pred.append(logits.argmax(dim=1).cpu().numpy())
        yv_pred = np.concatenate(yv_pred)
        val_macro_f1 = f1_score(y_va, yv_pred, average="macro")

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# =====================================================
# CSV APPEND (CRASH-SAFE)
# =====================================================
def append_row_to_csv(csv_path: str, row: dict):
    row_df = pd.DataFrame([row])

    if not os.path.exists(csv_path):
        row_df.to_csv(csv_path, index=False)
    else:
        row_df.to_csv(csv_path, mode="a", header=False, index=False)


# =====================================================
# BUILD PARAM COMBOS
# =====================================================
keys = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
param_combinations = [dict(zip(keys, combo)) for combo in itertools.product(*values)]
print(f"Total combos in grid: {len(param_combinations)}")


# =====================================================
# RUN SWEEP (RESUME-CAPABLE)
# =====================================================
log_f = open(LOG_PATH, "w", encoding="utf-8")
tee = Tee(sys.stdout, log_f)

try:
    with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
        completed = load_completed_param_ids(CSV_PATH)
        print(f"Resume: found {len(completed)} completed combos in existing CSV.")

        t0 = time.time()

        for idx, params in enumerate(param_combinations, 1):
            pid = stable_param_id(params)
            if pid in completed:
                print(f"[{idx}/{len(param_combinations)}] SKIP (done) param_id={pid} params={params}")
                continue

            print(f"\n[{idx}/{len(param_combinations)}] RUN param_id={pid} params={params}")

            row = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "param_id": pid,
                "use_balanced_weights": bool(USE_BALANCED_WEIGHTS),
                "seed": int(GLOBAL_SEED),
                **params
            }

            try:
                model = train_one_model(
                    X_train, y_train, X_val, y_val,
                    params,
                    class_weights=class_weights,
                    seed=GLOBAL_SEED
                )

                val_proba = predict_proba(model, X_val, batch_size=1024)
                val_pred  = val_proba.argmax(axis=1)
                val_metrics = compute_metrics_with_per_class(y_val, val_pred, val_proba, n_classes)

                test_proba = predict_proba(model, X_test, batch_size=1024)
                test_pred  = test_proba.argmax(axis=1)
                test_metrics = compute_metrics_with_per_class(y_test, test_pred, test_proba, n_classes)

                row.update({f"val_{k}": v for k, v in val_metrics.items()})
                row.update({f"test_{k}": v for k, v in test_metrics.items()})

                append_row_to_csv(CSV_PATH, row)
                completed.add(pid)

                print(
                    f"VAL  acc={row['val_acc']:.4f} bal_acc={row['val_bal_acc']:.4f} "
                    f"macro_f1={row['val_macro_f1']:.4f} mcc={row['val_mcc']:.4f}"
                )
                print(
                    f"TEST acc={row['test_acc']:.4f} bal_acc={row['test_bal_acc']:.4f} "
                    f"macro_f1={row['test_macro_f1']:.4f} mcc={row['test_mcc']:.4f}"
                )

                del model
                if device.type == "cuda":
                    torch.cuda.empty_cache()

            except Exception as e:
                row["error"] = str(e)
                append_row_to_csv(CSV_PATH, row)
                completed.add(pid)
                print(f"❌ FAILED param_id={pid} error={e}")

        print(f"\nSweep finished. Wall time: {time.time()-t0:.1f} sec")
        print(f"CSV saved incrementally at: {CSV_PATH}")

        # =====================================================
        # SELECT BEST + TRAIN AGAIN + SAVE ONLY BEST MODEL
        # =====================================================
        df = pd.read_csv(CSV_PATH)

        if "error" in df.columns:
            df_ok = df[df["error"].isna()]
        else:
            df_ok = df

        if df_ok.empty:
            raise RuntimeError("No successful runs found in CSV; cannot select best model.")

        if SELECT_BEST_BY not in df_ok.columns:
            raise RuntimeError(f"SELECT_BEST_BY='{SELECT_BEST_BY}' not found in CSV columns.")

        df_ok = df_ok.sort_values(SELECT_BEST_BY, ascending=False)
        best_row = df_ok.iloc[0].to_dict()

        best_params = {k: best_row[k] for k in keys}
        best_params["epochs"] = int(best_params["epochs"])
        best_params["patience"] = int(best_params["patience"])
        best_params["batch_size"] = int(best_params["batch_size"])
        best_params["use_bn"] = bool(best_params["use_bn"])
        best_params["lr"] = float(best_params["lr"])
        best_params["weight_decay"] = float(best_params["weight_decay"])
        best_params["dropout"] = float(best_params["dropout"])

        print("\n" + "=" * 90)
        print(f"BEST PARAMS by {SELECT_BEST_BY} = {best_row[SELECT_BEST_BY]:.6f}")
        print(best_params)
        print("=" * 90)

        print("\nTraining best model again (for saving)...")
        best_model = train_one_model(
            X_train, y_train, X_val, y_val,
            best_params,
            class_weights=class_weights,
            seed=GLOBAL_SEED
        )

        model_path = os.path.join(BEST_MODEL_DIR, "mlp_state_dict.pt")
        torch.save(best_model.state_dict(), model_path)

        # =====================================================
        # SAVE LATENT SPACE (.pt) FOR BEST MODEL
        # =====================================================
        test_is_synthetic = test["is_synthetic"].detach().cpu().numpy() if "is_synthetic" in test else None

        latent_test_path = os.path.join(BEST_MODEL_DIR, "test_latent_post_bn_relu_fc3.pt")
        save_latent_space_pt(
            model=best_model,
            X=X_test,
            y=y_test,
            device=device,
            save_path=latent_test_path,
            batch_size=1024,
            is_synthetic=test_is_synthetic,
            latent_name="post_bn_relu_fc3"
        )

        # =====================================================
        # SAVE PREDICTION OUTPUTS
        # =====================================================
        test_loader = make_loader(X_test, y_test, batch_size=1024, shuffle=False)

        y_true, y_pred, y_proba = collect_mlp_predictions(best_model, test_loader, device)

        save_prediction_outputs(
            save_dir=BEST_MODEL_DIR,
            y_true=y_true,
            y_pred=y_pred,
            y_proba=y_proba,
            class_names=[CLASS_ID_TO_NAME[i] for i in range(n_classes)],
            prefix="test"
        )

        meta = {
            "data_path": DATA_PATH,
            "results_csv": CSV_PATH,
            "select_best_by": SELECT_BEST_BY,
            "best_score": float(best_row[SELECT_BEST_BY]),
            "best_param_id": str(best_row["param_id"]),
            "best_params": best_params,
            "use_balanced_weights": bool(USE_BALANCED_WEIGHTS),
            "seed": int(GLOBAL_SEED),
            "feature_dim": int(input_size),
            "n_classes": int(n_classes),
            "architecture": {
                "hidden_sizes": [int(x) for x in HIDDEN_SIZES]
            },
            "class_id_to_name": {str(k): v for k, v in CLASS_ID_TO_NAME.items()},
            "latent_file": "test_latent_post_bn_relu_fc3.pt",
            "latent_name": "post_bn_relu_fc3",
            "timestamp_saved": ts,
        }
        meta_path = os.path.join(BEST_MODEL_DIR, "metadata.json")
        with open(meta_path, "w", encoding="utf-8") as jf:
            json.dump(meta, jf, indent=2)

        print(f"\n✅ Saved BEST model weights: {model_path}")
        print(f"✅ Saved BEST metadata: {meta_path}")
        print(f"✅ Saved BEST latent space: {latent_test_path}")

finally:
    log_f.close()
    print("Log file closed.")

# Confusion matrix plot

In [ ]:
# =====================================================
# 📊 CONFUSION MATRIX (FULLY CUSTOMIZABLE - SINGLE CELL)
# =====================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# =========================
# 🔧 USER INPUT
# =========================
NPZ_PATH = f"../Results/results_mlp_hparam_sweep/best_model/test_outputs.npz"   
SAVE_PATH = f"../Results/results_mlp_hparam_sweep/best_model/confusion_matrix.png"

# =========================
# 🎨 STYLE OPTIONS
# =========================
FONT_FAMILY = "DejaVu Sans"     
TITLE_SIZE = 16
LABEL_SIZE = 12
TICK_SIZE = 12
ANNOT_SIZE = 12

FIGSIZE = (6, 5)
DPI = 300

CMAP = "Blues"            # try: "viridis", "magma", "coolwarm"
SHOW_VALUES = True
FORMAT = ".2f"

NORMALIZE = True          # True = normalized, False = raw counts

# Grid / borders
SHOW_GRID = False
LINEWIDTH = 0.4
LINECOLOR = "black"

# Colorbar
SHOW_CBAR = True
CBAR_LABEL = ""

# Tick rotation
XTICK_ROT = 45
YTICK_ROT = 0

# =========================
# 📂 LOAD DATA
# =========================
data = np.load(NPZ_PATH, allow_pickle=True)

y_true = data["y_true"]
y_pred = data["y_pred"]
class_names = data["class_names"]

# =========================
# 📊 CONFUSION MATRIX
# =========================
cm = confusion_matrix(y_true, y_pred)

if NORMALIZE:
    cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)
    title = "Normalized Confusion Matrix"
    vmin, vmax = 0.0, 1.0
else:
    title = "Confusion Matrix"
    FORMAT = "d"
    vmin, vmax = None, None

# =========================
# 🎨 PLOT SETTINGS
# =========================
plt.rcParams["font.family"] = FONT_FAMILY

plt.figure(figsize=FIGSIZE, dpi=DPI)

ax = sns.heatmap(
    cm,
    annot=SHOW_VALUES,
    fmt=FORMAT,
    cmap=CMAP,
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=SHOW_CBAR,
    linewidths=LINEWIDTH if SHOW_GRID else 0,
    linecolor=LINECOLOR,
    vmin=vmin,
    vmax=vmax,
    annot_kws={"size": ANNOT_SIZE}
)

# Labels and title
plt.xlabel("Predicted Label", fontsize=LABEL_SIZE)
plt.ylabel("True Label", fontsize=LABEL_SIZE)
plt.title(title, fontsize=TITLE_SIZE)

# Tick formatting
plt.xticks(rotation=XTICK_ROT, ha="right", fontsize=TICK_SIZE)
plt.yticks(rotation=YTICK_ROT, fontsize=TICK_SIZE)

# Colorbar label
if SHOW_CBAR:
    cbar = ax.collections[0].colorbar
    cbar.set_label(CBAR_LABEL, fontsize=LABEL_SIZE)

plt.tight_layout()

# =========================
# 💾 SAVE (optional)
# =========================
if SAVE_PATH:
    plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches="tight")
    print(f"Saved figure → {SAVE_PATH}")

plt.show()

# ROC Curve plot

In [ ]:
# =====================================================
# ROC CURVE: per-class dotted lines + macro mean black
# with bootstrap std-dev shaded band
# =====================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# =========================
# USER INPUT
# =========================
NPZ_PATH = "../Results/results_mlp_hparam_sweep/best_model/test_outputs.npz"
SAVE_PATH = "../Results/results_mlp_hparam_sweep/best_model/roc_auc.png"

N_BOOTSTRAPS = 300   
RANDOM_SEED = 42

# =========================
# STYLE OPTIONS
# =========================
FONT_FAMILY = "DejaVu Sans"
FIGSIZE = (7, 6)
DPI = 600

TITLE_SIZE = 18
LABEL_SIZE = 16
TICK_SIZE = 14
LEGEND_SIZE = 10

MACRO_LINEWIDTH = 1.5
CLASS_LINEWIDTH = 1.5
STD_ALPHA = 0.25

MACRO_COLOR = "black"
STD_COLOR = "gray"
CLASS_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

# =========================
# LOAD DATA
# =========================
data = np.load(NPZ_PATH, allow_pickle=True)
y_true = data["y_true"]
y_proba = data["y_proba"]
class_names = data["class_names"]

n_classes = y_proba.shape[1]

# =========================
# BINARIZE LABELS
# =========================
y_bin = label_binarize(y_true, classes=np.arange(n_classes))

# =========================
# HELPERS
# =========================
def compute_per_class_roc(y_bin_local, y_score_local):
    fpr_dict, tpr_dict, auc_dict = {}, {}, {}
    for i in range(y_score_local.shape[1]):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_bin_local[:, i], y_score_local[:, i])
        auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])
    return fpr_dict, tpr_dict, auc_dict

def compute_macro_roc(fpr_dict, tpr_dict, n_cls):
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(n_cls)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_cls):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= n_cls
    macro_auc_val = auc(all_fpr, mean_tpr)
    return all_fpr, mean_tpr, macro_auc_val

# =========================
# BASE ROC
# =========================
fpr_base, tpr_base, auc_base = compute_per_class_roc(y_bin, y_proba)
macro_fpr, macro_tpr, macro_auc = compute_macro_roc(fpr_base, tpr_base, n_classes)

# =========================
# BOOTSTRAP FOR MACRO STD BAND
# =========================
rng = np.random.RandomState(RANDOM_SEED)
macro_tprs_boot = []

for _ in range(N_BOOTSTRAPS):
    idx = rng.choice(len(y_true), size=len(y_true), replace=True)

    # need all classes present for valid multiclass bootstrap
    if len(np.unique(y_true[idx])) < n_classes:
        continue

    y_true_bs = y_true[idx]
    y_proba_bs = y_proba[idx]
    y_bin_bs = label_binarize(y_true_bs, classes=np.arange(n_classes))

    try:
        fpr_bs, tpr_bs, _ = compute_per_class_roc(y_bin_bs, y_proba_bs)
        macro_fpr_bs, macro_tpr_bs, _ = compute_macro_roc(fpr_bs, tpr_bs, n_classes)

        interp_macro_tpr = np.interp(macro_fpr, macro_fpr_bs, macro_tpr_bs)
        interp_macro_tpr[0] = 0.0
        interp_macro_tpr[-1] = 1.0
        macro_tprs_boot.append(interp_macro_tpr)
    except Exception:
        continue

macro_tprs_boot = np.array(macro_tprs_boot)

if len(macro_tprs_boot) > 0:
    macro_mean_tpr = macro_tprs_boot.mean(axis=0)
    macro_std_tpr = macro_tprs_boot.std(axis=0)
    macro_lower = np.maximum(macro_mean_tpr - macro_std_tpr, 0)
    macro_upper = np.minimum(macro_mean_tpr + macro_std_tpr, 1)
else:
    macro_mean_tpr = macro_tpr.copy()
    macro_lower = macro_tpr.copy()
    macro_upper = macro_tpr.copy()

# =========================
# PLOT
# =========================
plt.rcParams["font.family"] = FONT_FAMILY
plt.figure(figsize=FIGSIZE, dpi=DPI)

# per-class dotted ROC curves
for i in range(n_classes):
    plt.plot(
        fpr_base[i],
        tpr_base[i],
        linestyle=":",
        linewidth=CLASS_LINEWIDTH,
        color=CLASS_COLORS[i % len(CLASS_COLORS)],
        label=f"{class_names[i]} (AUC = {auc_base[i]:.3f})"
    )

# macro mean solid black line
plt.plot(
    macro_fpr,
    macro_tpr,
    color=MACRO_COLOR,
    linewidth=MACRO_LINEWIDTH,
    label=f"Macro-average (AUC = {macro_auc:.3f})"
)

# macro std-dev shaded band
plt.fill_between(
    macro_fpr,
    macro_lower,
    macro_upper,
    color=STD_COLOR,
    alpha=STD_ALPHA,
    label="Macro ±1 std"
)

# random baseline
plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)

plt.xlabel("False Positive Rate", fontsize=LABEL_SIZE)
plt.ylabel("True Positive Rate", fontsize=LABEL_SIZE)
plt.title("ROC Curve: Macro-Average and Per-Class", fontsize=TITLE_SIZE)

plt.xticks(fontsize=TICK_SIZE)
plt.yticks(fontsize=TICK_SIZE)
plt.legend(fontsize=LEGEND_SIZE, loc="lower right", frameon=True)
plt.grid(alpha=0.0)

plt.tight_layout()

if SAVE_PATH is not None:
    plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches="tight")
    print(f"Saved → {SAVE_PATH}")

plt.show()